In [1]:
# Install BERTopic
%pip install bertopic

# Choose an embedding backend
%pip install bertopic[flair, gensim, spacy, use]

# Topic modeling with images
%pip install bertopic[vision]

Note: you may need to restart the kernel to use updated packages.
zsh:1: bad pattern: bertopic[flair,
Note: you may need to restart the kernel to use updated packages.
zsh:1: no matches found: bertopic[vision]
Note: you may need to restart the kernel to use updated packages.


Extract topics from the well-known 20 newsgroups dataset

In [2]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
docs = docs[:2000]

Clean the text

In [3]:
import re # regular expression module : search, replace and remove patterns in text

def clean(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", " ", text)          
    # remove URLs
    text = re.sub(r"\b\w+@\w+\.\w+\b", " ", text)          
    # remove emails
    text = re.sub(r"\[deleted\]|\[removed\]", " ", text, flags=re.I)
    text = re.sub(r"[\*\_`>#]", " ", text)                 
    # light markdown chars
    text = re.sub(r"\s+", " ", text)                       
    # collapse whitespace
    return text.strip()
    # removes leading or trailing spaces

cleaned_docs = []

for d in docs:
    if isinstance(d, str) and len(d) > 50:
        cleaned_docs.append(clean(d))

docs = cleaned_docs

Fit BASE model
- note that the embedding (converting text into vectors), dimensionality reduction, and clustering steps were not explicitly written 
- fit_transform() automatically ran UMAP (dimensionality reduction) and HDBSCAN (clustering) c-TF-IDF words (representation method) by default

In [4]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

base_model = BERTopic(
    min_topic_size=10,
    umap_model=UMAP(n_neighbors=10, min_dist=0.0),
    hdbscan_model=HDBSCAN(min_cluster_size=10)
)

base_topics, _ = base_model.fit_transform(docs)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Access the frequent topics that were generated
- Top keywords that define the topic [the, to, of, ...] show that it is dominated by common words (i.e. not meaningful) -> stopwords weren't removed strongly for topic representation / need to tune the vectoriser / representation

In [5]:
base_model.get_topic_info().head(15)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,21,-1_the_to_04_and,"[the, to, 04, and, won, in, lost, that, 05, of]","[Ah, so you finally found a use for that super..."
1,0,1732,0_the_to_of_and,"[the, to, of, and, is, in, that, it, for, you]",[THE WHITE HOUSE Office of the Press Secretary...
2,1,83,1_the_to_and_in,"[the, to, and, in, of, he, that, was, 10, is]",[Perhaps it was the most exciting game played ...
3,2,50,2_the_he_to_in,"[the, he, to, in, and, that, is, year, of, his]",[Not particularly in the World Series. During ...
4,3,31,3_the_to_and_games,"[the, to, and, games, game, on, in, is, espn, at]",[} } [Stuff about upated inning scores deleted...


Add proper vectoriser (stopwords + bigrams + min_df)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

Refit with better representation model
- Vectoriser removes English stopwords + allows bigrams -> effects keyword extraction quality
- Representation model (KeyBERTInspired) re-ranks topic words using embedding similarity
- Smaller clusters are discouraged / merged because of min topic size

In [7]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

improved_model = BERTopic(
    vectorizer_model=vectorizer_model,
    representation_model=KeyBERTInspired(),
    min_topic_size=10
)

topics, _ = improved_model.fit_transform(docs)
improved_model.get_topic_info()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Topic,Count,Name,Representation,Representative_Docs
0,0,1730,0_bus_hardware_drivers_driver,"[bus, hardware, drivers, driver, armenians, ca...","[Let's face it, if the words don't get into yo..."
1,1,187,1_replay_espn_montreal_flyers,"[replay, espn, montreal, flyers, red wings, pu...",[NHL PLAYOFF RESULTS FOR 4/19/93. ------------...
